<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
Copyright (c) 2026 Intel Corporation
SPDX-License-Identifier: Apache-2.0

NOTE: This file has been modified by Intel Corporation.
-->

# Module wrapper demo
The purpose of this notebook is to walk through the Torch Tweak module wrapper usage.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd ..

In [ ]:
import os
from logging import basicConfig

import torch

from torch_tweak.global_context import BATCH_SIZE_KEY, global_context
from torch_tweak.torch.backend.torch_inductor_backend import TorchInductorBackend
from torch_tweak.torch.module.wrapper_module import Module
from torch_tweak.torch.tune_strategy.one_backend_strategy import OneBackendStrategy

log_level = os.environ.get("TORCH_TWEAK_LOG_LEVEL", "INFO")
basicConfig(level=log_level, format="%(asctime)s - %(levelname)s - %(message)s", force=True)

## Simple identity model
Let's start with something simple.

In [ ]:
class Identity(torch.nn.Module):
    """Takes one input and returns it."""

    def forward(self, x, **kwargs):
        return x

#### Let's wrap module

In [ ]:
model = Identity()
module = Module(model, "demo-identity")
device = torch.device("xpu" if torch.xpu.is_available() else "cpu")
print(f"Using device: {device}")

#### Record some samples

If you call a wrapped module it will save sample input and output data of a model.

The following is an example how to call a model and record data.

In [ ]:
module(torch.tensor(1), a=True)

Apart from saving data (which is used later on during tuning) it will also create graph specifications.

A `GraphSpec` has a `name` and corresponding 
- `input_spec` - a way of describing model inputs: `args` and `kwargs`
- `output_spec` - a way of describing model outputs

Let's look at the `graph_spec`.

In [ ]:
print(module.graph_specs[0])

The `input_spec` shows one tensor as the first argument `args_0`. The `Locator` is a recipe how to extract this tensor from recorded data.

Apart from the tensor we have also `other` data i.e. one keyword argument `kwargs_a`.

Based on the `input_spec` Torch Tweak tries to create different `graph_specs` (i.e. the `output_spec` does not play role here).

Let's record another sample and see if the number of graphs changed.

In [ ]:
print("Number of graphs before second sample: ", len(module.graph_specs))
module(torch.tensor(2), a=True)
print("Number of graphs after second sample: ", len(module.graph_specs))

Since the tensor rank did not change (only its value) - there is no new graph created. However, if the other arguments change, or the rank,
 there will be a new graph_spec created.

 Let's change the **kwarg** argument.

In [ ]:
print("Number of graphs before second sample: ", len(module.graph_specs))
module(torch.tensor(2), a=False)
print("Number of graphs after second sample: ", len(module.graph_specs))

Let's record a sample with a different rank of a tensor.

In [ ]:
module(torch.randn(1, 1))

# print all graph specs
for gs in module.graph_specs:
    print("-" * 100)
    print(gs)

As can be seen, third graph has tensor as input with shapes `[1, 1]` and min and max shapes being the same.

If we record more samples of different shapes, the graph will have dynamic dimensions calculated.

In [ ]:
module(torch.randn(2, 2))
module.graph_specs[-1]

As can be seen, dynamic shapes are named `dim0`, `dim1` - they were calculated because we recorded same tensor with different shapes meaning they have to be dynamic.

The above also shows that graph has seen min and max shapes of `[1, 1]` and `[2, 2]` respectively i.e. first dimension is from 1 to 2, same for the second.

See that the module is in the `recording` state:

In [ ]:
module.state

#### Automatic batch dimension detection

As we have seen module detects dynamic axes just by looking at different input data. However for checking performance and sanity Torch Tweak needs to detect batch axis - which is special type of dynamic axis i.e. grows with a batch size. One example could be LLM pipeline where we have batch and sequence length dynamic axes `B` and `L` but only `B` grows with a batch size.

In order to detect batch dimension Module wrapper has to have information about current global batch size. This is because the `global batch size` can be different than `local batch size` e.g. SDXL model stacks input tensor and with `global bs=1` a module can receive `local bs=2`.

When you perform tuning, the Torch Tweak takes dataloader and populates information about `batch size` so that module wrapper can obtain it. Since we are demonstrating module wrapper in isolation, we have to provide that information ourselves.

In [ ]:
with global_context:
    global_context[BATCH_SIZE_KEY] = 1
    module(torch.randn(1, 2, 3))
    global_context[BATCH_SIZE_KEY] = 2
    module(
        torch.randn(
            2,
            4,
            3,
        )
    )

In [ ]:
module.graph_specs[-1]

As can be seen, the last graph has batch axis `batch0` which grows with batch size and `batch1` which grows twice the batch size.

#### Check changing state to passthrough and back to recording.

You can set a wrapped module to "passthrough" mode, which disables sample recording so that the graph remains unchanged.

In [ ]:
print("Number of graphs before a passthrough sample: ", len(module.graph_specs))

module.enable_passthrough()
print(module.state)
module(torch.randn(1, 2, 3), showcase_passthrough=True)

print("Number of graphs after the passthrough sample: ", len(module.graph_specs))

module.enable_recording()
print(module.state)
module(torch.randn(1, 2, 3), showcase_passthrough=False)

print("Number of graphs after recording sample: ", len(module.graph_specs))


#### Let's try tune dry-run

In [ ]:
module.tune(dry_run=True, device=device)

# Resnet50

In [ ]:
# example from https://huggingface.co/docs/timm/en/models/resnet
import urllib
from pathlib import Path

import timm
from PIL import Image
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

In [ ]:
model = timm.create_model("resnet50", pretrained=True).to("xpu").eval()
config = resolve_data_config({}, model=model)
transform = create_transform(**config)

url, filename = ("https://github.com/pytorch/hub/raw/master/images/dog.jpg", "dog.jpg")
if not Path(filename).exists():
    urllib.request.urlretrieve(url, filename)

img = Image.open(filename).convert("RGB")
data = transform(img).unsqueeze(0).to("xpu")  # transform and add batch dimension
data.shape

In [ ]:
with torch.no_grad():
    out = model(data)
ref_probs = torch.nn.functional.softmax(out[0], dim=0)

#### Let's wrap module

In [ ]:
module = Module(model, "demo-resnet50")

#### Record some samples

In [ ]:
with global_context as ctx:
    ctx[BATCH_SIZE_KEY] = 1
    _ = module(data)

module.graph_specs[0]

In [ ]:
# Record bs = 2 to have dynamic batch dimension
with global_context as ctx:
    ctx[BATCH_SIZE_KEY] = 2
    _ = module(data.repeat(2, 1, 1, 1))

In [ ]:
module.graph_specs[0]

#### Let's try tune dry-run

In [ ]:
torch_compile_strategy = OneBackendStrategy(TorchInductorBackend())

In [ ]:
module.tune(strategy=torch_compile_strategy, dry_run=True, device=device)

#### Let's try real tune with torch compile

In [ ]:
module.tune(strategy=torch_compile_strategy, dry_run=False, device=device)

In [ ]:
module.state
out = module(data)
actual_probs = torch.nn.functional.softmax(out[0], dim=0)

In [ ]:
module.state

In [ ]:
torch.testing.assert_close(ref_probs, actual_probs, rtol=1e-4, atol=1e-4)